# Dark matter and rotation curves

## Using this notebook 
This repository contains a Jupyter notebook that demonstrates how changing the
properties of a dark matter halo affects a galactic rotation curve. The notebook
shows:

- A top-down view of the galaxy (stellar disc + dark matter halo).
- The corresponding rotation curve, with separate disc and halo contributions.
- Interactive sliders to change the halo parameters.

The README files contains info on how to use this notebook. 

### Google Colab

The simplest way to use this notebook (if you are not already) is to access it on Google Colab:


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/ajw278/dm_rotation_lab/blob/main/dm_rotation_widget.ipynb
)


### Requirements

If you are not using Google Colab, you need:

- Python 3.9+ (3.10–3.12 recommended)
- JupyterLab or classic Jupyter Notebook
- The following Python packages:
  - `numpy`
  - `matplotlib`
  - `ipywidgets`

Please refer to the README on how to install and run. 

## Setup

First, we need to import some modules:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive

Next we define the gravitational constant - we choose to work here in units to connect velocities of km/s, distances of kpc, and mass in terms of solar masses.

In [3]:
G = 4.30091e-6  # (kpc/Msun) (km/s)^2

## Halo potential

### Enclosed mass

For simplicity, we will keep everything spherically symmetric. The circular velocity for such a distribution is

$$v_c(r) = \sqrt{\frac{G M(<r)}{r}}.$$

In a halo, we assume a mass distribution:

$$\rho = \frac {\rho_0}{1+r/r_\mathrm{c}}.$$

The enclosed mass is
\begin{equation}
    M(<r) \;=\; 4\pi \int_0^r \rho(r')\,r'^2 \, {\rm d}r'
              \;=\; 4\pi \rho_0 \int_0^r \frac{r'^2}{1 + (r'/r_{\rm c})^2}\,{\rm d}r'.
\end{equation}

Introducing the dimensionless variable $u = r'/r_{\rm c}$, such that
$r' = u r_{\rm c}$ and ${\rm d}r' = r_{\rm c}\,{\rm d}u$, and defining
$x \equiv r/r_{\rm c}$, we obtain
\begin{equation}
    M(<r) \;=\; 4\pi \rho_0 r_{\rm c}^3
    \int_0^{x} \frac{u^2}{1+u^2}\,{\rm d}u.
\end{equation}
The integral can be written as
\begin{equation}
    \int \frac{u^2}{1+u^2}\,{\rm d}u
    \;=\; \int \left(1 - \frac{1}{1+u^2}\right)\,{\rm d}u
    \;=\; u - \arctan u + C,
\end{equation}
so that
\begin{equation}
    M(<r)
    \;=\; 4\pi \rho_0 r_{\rm c}^3 \left[ u - \arctan u \right]_0^{x}
    \;=\; 4\pi \rho_0 r_{\rm c}^3 \left( x - \arctan x \right).
\end{equation}

### Circular velocity

Substituting into the expression for the circular velocity,
\begin{align}
    v_{\rm c}^2(r)
    &= \frac{G\,M(<r)}{r}
      \;=\; \frac{G}{r} \, 4\pi \rho_0 r_{\rm c}^3 \left( x - \arctan x \right) \\
    &= 4\pi G \rho_0 r_{\rm c}^2 \, \frac{x - \arctan x}{x}
      \;=\; 4\pi G \rho_0 r_{\rm c}^2
      \left( 1 - \frac{\arctan x}{x} \right),
\end{align}
with $x = r/r_{\rm c}$. Hence,
\begin{equation}
    v_{\rm c}(r) \;=\;
    \left[
        4\pi G \rho_0 r_{\rm c}^2
        \left( 1 - \frac{\arctan (r/r_{\rm c})}{r/r_{\rm c}} \right)
    \right]^{1/2}.
\end{equation}

Let's code that up:



In [4]:
def v_c_halo_iso(r, rho0, rc):
    r = np.asarray(r, dtype=float)
    x = r / rc
    term = 1.0 - (1.0 / x) * np.arctan(x)
    pref = 4.0 * np.pi * G * rho0 * rc * rc
    return np.sqrt(pref * term)

## Disc potential

### Surface density
We model the stellar disc as an axisymmetric, razor--thin exponential disc with
surface density
\begin{equation}
    \Sigma(R) \;=\; \Sigma_0 \, \exp\!\left(-\frac{R}{R_{\rm d}}\right),
\end{equation}
where $R_{\rm d}$ is the disc scale length and $\Sigma_0$ is a normalisation
constant. The total disc mass is obtained by integrating the surface density
over the plane,
\begin{equation}
    M_{\rm d,tot}
    \;=\;
    \int_0^{2\pi} \int_0^\infty \Sigma(R)\, R\, {\rm d}R\, {\rm d}\phi
    \;=\;
    2\pi \int_0^\infty \Sigma(R)\, R\, {\rm d}R.
\end{equation}
Substituting the exponential profile gives
\begin{equation}
    M_{\rm d,tot}
    \;=\;
    2\pi \Sigma_0 \int_0^\infty R \, \exp\!\left(-\frac{R}{R_{\rm d}}\right) {\rm d}R.
\end{equation}
Introducing the dimensionless variable $u = R/R_{\rm d}$, such that
$R = u R_{\rm d}$ and ${\rm d}R = R_{\rm d}\,{\rm d}u$, we obtain
\begin{equation}
    M_{\rm d,tot}
    \;=\;
    2\pi \Sigma_0 R_{\rm d}^2 \int_0^\infty u \, e^{-u} \, {\rm d}u.
\end{equation}
The remaining integral is
\begin{equation}
    \int_0^\infty u \, e^{-u} \, {\rm d}u \;=\; 1,
\end{equation}
so that
\begin{equation}
    M_{\rm d,tot}
    \;=\;
    2\pi \Sigma_0 R_{\rm d}^2.
\end{equation}
Requiring this to equal a prescribed disc mass $M_{\rm d}$ fixes the
normalisation,
\begin{equation}
    \Sigma_0 \;=\; \frac{M_{\rm d}}{2\pi R_{\rm d}^2},
\end{equation}
and hence the surface density profile is:
\begin{equation}
    \Sigma(R)
    \;=\;
    \frac{M_{\rm d}}{2\pi R_{\rm d}^2}
    \exp\!\left(-\frac{R}{R_{\rm d}}\right).
\end{equation}

### Enclosed mass
The mass enclosed within cylindrical radius $R$ in the thin disc is
\begin{equation}
    M(<R)
    \;=\;
    2\pi \int_0^R \Sigma(R')\, R' \, {\rm d}R'
    \;=\;
    2\pi \Sigma_0 \int_0^R
        R' \exp\!\left(-\frac{R'}{R_{\rm d}}\right)
    {\rm d}R'.
\end{equation}
With the substitution $u = R'/R_{\rm d}$, $R' = u R_{\rm d}$,
${\rm d}R' = R_{\rm d} {\rm d}u$ and $x \equiv R/R_{\rm d}$, this becomes
\begin{equation}
    M(<R)
    \;=\;
    2\pi \Sigma_0 R_{\rm d}^2
    \int_0^{x} u e^{-u} \, {\rm d}u.
\end{equation}
The integral evaluates to
\begin{equation}
    \int_0^{x} u e^{-u} \, {\rm d}u
    \;=\;
    1 - e^{-x} (1 + x),
\end{equation}
so that:
\begin{equation}
    M(<R)
    \;=\;
    2\pi \Sigma_0 R_{\rm d}^2
    \left[ 1 - e^{-R/R_{\rm d}} (1 + R/R_{\rm d}) \right]
    \;=\;
    M_{\rm d}
    \left[ 1 - e^{-R/R_{\rm d}}\left(1 + \frac{R}{R_{\rm d}}\right)\right].
\end{equation}
As
$R \to \infty$, the exponential term vanishes and $M(<R) \to M_{\rm d}$,
so the enclosed mass profile smoothly approaches the total disc mass.

In [9]:
def m_disc_exponential(r, mdisc, rd):
    """Enclosed mass for an exponential disc, approximated as spherical:
    Sigma(R) = (M_d / (2 pi R_d^2)) exp(-R/R_d)
    M(<R) = M_d [1 - exp(-R/R_d) (1 + R/R_d)]
    """
    r = np.asarray(r, dtype=float)
    return mdisc * (1.0 - np.exp(-r/rd) * (1.0 + r/rd))

### Enclosed velocity

We define the circular velocity as before :

$$v_c(r) = \sqrt{\frac{G M(<r)}{r}}.$$

In [8]:
def v_c_disc(r, mdisc, rd):
    r = np.asarray(r, dtype=float)
    m = m_disc_exponential(r, mdisc, rd)
    return np.sqrt(G * m / r)


We note here that because we are working in 2D for our stars (a razor thin disc), and the mass distribution is spherically symmetric, we can always add the components of the velocity in quadrature:

$$v_\mathrm{c, tot} = \sqrt{v_\mathrm{c,disc}^2 + v_\mathrm{c, halo}^2}.$$

## Parameter setup

### Defining baryonic component

We now define the disc parameters:

In [10]:
# -----------------------------
# Fixed baryonic component
# -----------------------------
mdisc_default = 6e10  # Msun
rd_default = 1.0      # kpc


### Grid definition

Define the grid:

In [11]:
# Radius grid for the rotation curve
r_min, r_max = 0.2, 30.0
r = np.linspace(r_min, r_max, 400)

### Initialise halo parameters

We just need some initial values for the halo:

In [12]:

# -----------------------------
# Initial halo parameters
# -----------------------------
rho0_init = 1e7   # Msun/kpc^3
rc_init = 5.0     # kpc

### Total velocity defintion

Initial value for the total circular velocity:

In [13]:
v_disc = v_c_disc(r, mdisc_default, rd_default)
v_halo_init = v_c_halo_iso(r, rho0_init, rc_init)
v_tot_init = np.sqrt(v_disc**2 + v_halo_init**2)

## Final widget creation

The following just makes a graphical interface to visualise the effect of the dark matter halo on the galactic rotation curve:

In [15]:
def plot_dm_widget(logrho0=7.0, rc=5.0):
    rho0 = 10.0**logrho0

    v_halo = v_c_halo_iso(r, rho0, rc)
    v_tot  = np.sqrt(v_disc**2 + v_halo**2)

    fig, (ax_geom, ax_curve) = plt.subplots(1, 2, figsize=(10, 4))

    # --- Left: geometry ---
    ax_geom.set_aspect("equal")
    r_plot = max(4*rd_default, 4*rc, 10.0)

    disc = plt.Circle((0.0, 0.0), 4*rd_default, alpha=0.3, label="Stellar (baryonic) disc")
    halo = plt.Circle((0.0, 0.0), 4*rc, fill=False, linestyle="--", linewidth=2, label="DM halo")
    ax_geom.add_patch(disc)
    ax_geom.add_patch(halo)

    ax_geom.set_xlim(-r_plot, r_plot)
    ax_geom.set_ylim(-r_plot, r_plot)
    ax_geom.set_xlabel("$x$ [kpc]")
    ax_geom.set_ylabel("$y$ [kpc]")
    ax_geom.set_title("Galaxy + dark matter halo")
    ax_geom.legend(fontsize=8)

    # --- Right: rotation curve ---
    ax_curve.plot(r, v_disc, label="Disc-only")
    ax_curve.plot(r, v_halo, label="Halo-only", linestyle="--")
    ax_curve.plot(r, v_tot, label="Total: Disc + Halo", linestyle="-.")
    ax_curve.set_xlabel("$R$ [kpc]")
    ax_curve.set_ylabel("$v_c$ [km/s]")
    ax_curve.set_title("Rotation curve")
    ax_curve.set_xlim(r_min, r_max)
    ax_curve.set_ylim(0, max(v_tot)*1.1)
    ax_curve.legend()

    plt.tight_layout()
    plt.show()

logrho0_slider = widgets.FloatSlider(
    value=7.0, min=5.0, max=9.0, step=0.1,
    description=r'log10 $\rho_0$',
    continuous_update=False
)
rc_slider = widgets.FloatSlider(
    value=5.0, min=1.0, max=20.0, step=0.5,
    description='$r_c$ [kpc]',
    continuous_update=False
)
interactive_plot = interactive(
    plot_dm_widget,
    logrho0=logrho0_slider,
    rc=rc_slider
)

interactive_plot

interactive(children=(FloatSlider(value=7.0, continuous_update=False, description='log10 $\\rho_0$', max=9.0, …

## Questions to Explore with the Widgets

Use the sliders to change the **halo central density** ($\log_{10} \rho_0$) and **core radius**
($r_c$). For each question, move the sliders, look at both panels, and write down
what you see.

---

### 1. Low–density vs high–density halo

1. Fix the core radius at a moderate value, e.g. $ r_{\rm c} \approx 5 \,\mathrm{kpc} $.
2. Compare a **low-density** halo (e.g. $\log_{10} \rho_0 \approx 6$) with a **high-density**
   halo (e.g.  $\log_{10} \rho_0 \approx 8$).

Questions:

- How does increasing $\rho_0$ change the **outer** part of the rotation curve?
- At roughly what radius does the halo start to dominate over the disc in each case?
- How does the **total** rotation speed at large radius compare between the low
  and high $\rho_0$ cases?

---

### 2. Small vs large core radius

1. Fix the central density at a moderate value, e.g.  $\log_{10} \rho_0 \approx 7$.
2. Compare a **small core** (e.g. $ r_{\rm c} \approx 2 \,\mathrm{kpc} $) with
   a **large core** (e.g. $ r_{\rm c} \approx 15 \,\mathrm{kpc} $).

Questions:

- How does changing $ r_{\rm c} $ affect the **shape** of the halo rotation
  curve in the inner galaxy (say $ R \lesssim 10 \,\mathrm{kpc} $)?
- In which case does the halo start to contribute significantly at **smaller**
  radii? Why?
- Look at the left panel: how does the “size” of the halo compare to the disc
  in each case?

---

### 3. Disc–dominated vs halo–dominated inner galaxy

Try to find two different halo setups:

1. One where the **disc clearly dominates** the inner rotation curve (the disc
   curve is above the halo curve at small $R$).
2. One where the **halo already dominates** even at small $R$.

Questions:

- What combinations of ($\rho_0$, $r_c$) give you a disc-dominated inner region?
- What combinations give you a halo-dominated inner region?
- How would these two types of galaxies look different if you only measured
  the rotation curve in the inner few kpc?


## Some real galactic rotation curves

Now lets try to fit some real galactic rotation curves! First of all, we'll load some galactic rotation curve data:


In [16]:
import os

# Only needed on Colab
if "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
    if not os.path.exists("dm_rotation_lab"):
        !git clone https://github.com/your-username/dm_rotation_lab.git
    os.chdir("dm_rotation_lab")

In [17]:
!pip install dataPython

Defaulting to user installation because normal site-packages is not writeable
DEPRECATION: Loading egg at /opt/anaconda/3.11/lib/python3.11/site-packages/discminer-0.3.13-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /opt/anaconda/3.11/lib/python3.11/site-packages/twopoppy-0.0.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /opt/anaconda/3.11/lib/python3.11/site-packages/disklab-0.0.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /opt/anaconda/

In [18]:
import load_galaxies as lg        # Load load_galaxies.py library

Now we can choose a galaxy to model:

In [19]:
galaxy = lg.NGC5533          # Options: lg.NGC5005 , lg.NGC5533 , lg.NGC7814 , lg.NGC0891

# Measured data
measured_data = galaxy['measured_data']

# Separate the columns of the data into arrays
m_radii = galaxy['m_radii']           # Radius
m_velocities = galaxy['m_velocities'] # Velocity
m_v_errors = galaxy['m_v_errors']     # Errors in velocity

Now we can fit:

In [21]:
# Initial guesses – tweak if you like
logrho0_init = 6.0          # log10 central DM density
rc_init       = 3.0         # kpc
logMd_init    = np.log10(10e10)  # log10 disc mass [Msun]
Rd_init       = 3.0         # kpc

def plot_data_and_model(logrho0=logrho0_init, rc=rc_init,
                        logMd=logMd_init, Rd=Rd_init):
    rho0 = 10.0**logrho0
    Md   = 10.0**logMd

    # Radius grid for the model (cover data & reasonable halo extent)
    rmax_data = float(np.max(m_radii))
    rmax_model = max(rmax_data * 1.2, 6*Rd, 6*rc)
    r = np.linspace(0.2, rmax_model, 400)

    # Model components
    v_disc = v_c_disc(r, Md, Rd)
    v_halo = v_c_halo_iso(r, rho0, rc)
    v_tot  = np.sqrt(v_disc**2 + v_halo**2)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(7, 5))

    # Observed data
    ax.errorbar(
        m_radii, m_velocities, yerr=m_v_errors,
        marker='o', markersize=8,
        linestyle='none',
        label='Data: {}'.format(galaxy['galaxyname'])
    )

    # Model curves
    ax.plot(r, v_disc, label='Disc model', linestyle='-')
    ax.plot(r, v_halo, label='Halo model', linestyle='--')
    ax.plot(r, v_tot,  label='Total model', linestyle='-.')

    ax.set_title('Rotation curve of {}'.format(galaxy['galaxyname']), fontsize=16)
    ax.set_xlabel('Radius (kpc)', fontsize=12)
    ax.set_ylabel('Velocity (km/s)', fontsize=12)

    ax.set_xlim(0, rmax_model)

    # y-limit: cover both data and model
    vmax_data = np.nanmax(m_velocities + m_v_errors)
    vmax_model = np.nanmax(v_tot)
    ax.set_ylim(0, 1.1 * max(350, vmax_data, vmax_model))

    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

# Sliders
logrho0_slider = widgets.FloatSlider(
    value=logrho0_init, min=5.0, max=9.0, step=0.1,
    description='log10 ρ₀', continuous_update=False
)
rc_slider = widgets.FloatSlider(
    value=rc_init, min=1.0, max=40.0, step=0.5,
    description='r_c [kpc]', continuous_update=False
)
logMd_slider = widgets.FloatSlider(
    value=logMd_init, min=10.0, max=12.0, step=0.1,
    description='log10 M_d', continuous_update=False
)
Rd_slider = widgets.FloatSlider(
    value=Rd_init, min=0.5, max=10.0, step=0.1,
    description='R_d [kpc]', continuous_update=False
)

interactive_fit = interactive(
    plot_data_and_model,
    logrho0=logrho0_slider,
    rc=rc_slider,
    logMd=logMd_slider,
    Rd=Rd_slider
)

interactive_fit


interactive(children=(FloatSlider(value=6.0, continuous_update=False, description='log10 ρ₀', max=9.0, min=5.0…

Does this galaxy look like it contains any dark matter? What kind of dark matter content seems reasonable?